# Chapter 10 — Rebase & Cherry-pick
> *Two surgical tools for people who care about clean history.*
> *Seniors love rebase. Juniors fear rebase. This chapter makes you a senior.* 

---

>  Git history is like your commit messages — it tells a story.
>
> With merge: "We merged feature/login into main. Here's the join point."
> With rebase: "There was never a join point. The feature always grew from latest main."
>
> Merge is honest. Rebase is revisionist history.
> Both are useful. Knowing WHEN to use each is the skill.

##  What is Rebase? The Metaphor That Actually Works

```
Before rebase:                 After rebase:
ABC  (main)                  ABC  (main)
    \                                   \
     DE  (feature)                      D'E'  (feature, moved ON TOP of C)
```

Merge would have created F (a merge commit).
Rebase takes D and E, replays them as if they started from C.
Result: clean, linear history. No merge commit. Just a straight line of commits.

>  Rebase is like telling your friends:
> "I didn't write this code while the project was at version D and E.
> I wrote it after version C."
>
> It's a lie. But it's a very clean, useful lie.
> The code works exactly the same. The history looks prettier.
> Open source project maintainers LOVE clean history.

##  `git rebase` Commands

```bash
# Standard rebase — move your feature branch on top of main's latest commit:
git checkout feature/my-thing
git rebase main

# Conflict during rebase? Fix the file, then:
git add <fixed-file>
git rebase --continue

# OR completely bail out and go back to where you started:
git rebase --abort

# After rebasing, your local history was rewritten — force-push is required:
git push --force-with-lease origin feature/my-thing
# (Use --force-with-lease, not --force — see the Push chapter for why)
```

>  **The cardinal rule of rebase:**
> **Never rebase `main`, `develop`, or any branch other people are working on.**
>
> Rebase rewrites commit hashes. If teammates have those old hashes, their histories now conflict.
> Their `git pull` will give them a headache. They will come to you.
> The conversation will not be pleasant.
>
> Rebase is ONLY safe on YOUR OWN private feature branches that nobody else has pulled.

##  Interactive Rebase — Clean Up Your Commits Before a PR

This is the most powerful thing in Git. You get to edit history before anyone sees it.

```bash
git rebase -i HEAD~4    # Edit the last 4 commits
# Opens your editor with this view:
# pick abc1234 Add login form
# pick def5678 Fix typo
# pick aaa9999 WIP: still broken lol
# pick bbb0000 ACTUALLY fix login

# Change 'pick' to:
# squash (s) — combine this commit with the one above it
# fixup  (f) — like squash but throw away this commit's message
# reword (r) — keep the commit, but edit its message
# drop   (d) — delete this commit completely (gone forever)
# edit   (e) — pause here so you can amend this specific commit

# Example — squash 4 messy commits into 1 clean commit:
# pick abc1234 Add login form
# fixup def5678 Fix typo
# fixup aaa9999 WIP: still broken lol
# fixup bbb0000 ACTUALLY fix login
# → Save and close editor
# → 4 commits became 1: "Add login form"
# → Your PR looks professional. Nobody sees the WIP shame. 
```

>  The professional standard before opening a PR:
> Your branch might have 8 commits of "WIP", "trying", "fix", "okk this time".
> Interactive rebase squashes them into 1-3 meaningful commits before review.
> The reviewer sees clean, logical steps — not your live debugging session.
>
> It's not dishonesty. It's editing. Authors edit. Developers edit. Use it.

##  `git cherry-pick` — Take Exactly ONE Commit

Cherry-pick = take a specific commit from any branch and apply it to your current branch.
Like reaching into a tree and grabbing the one apple you want. 

```bash
# Find the commit hash you want:
git log --oneline other-branch

# Apply that commit to your current branch:
git cherry-pick <hash>

# Apply multiple commits:
git cherry-pick <hash1> <hash2>

# Apply a range of commits:
git cherry-pick <start-hash>^..<end-hash>

# Conflict during cherry-pick? Fix it, then:
git add <fixed-file>
git cherry-pick --continue
# Or abort entirely:
git cherry-pick --abort
```

**When to reach for cherry-pick:**
- Bug fix is on a feature branch. Production needs that fix RIGHT NOW before the feature is ready.
- You accidentally committed to the wrong branch. Pick it to the right one.
- Backporting a fix from `main` to an older release branch (v1.x still has users).

>  Cherry-pick feels like cheating.
> "I'll just take this ONE commit from that branch."
> It's not cheating. It's targeted. It's practical.
> Just don't overuse it — if you're cherry-picking 15 commits,
> maybe you should rethink your branching strategy.